### Truncation experiment

In [1]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
# register and use Arial font in Matplotlib figures
font_manager.fontManager.addfont('ARIAL.TTF')
# Set global plotting defaults
plt.rcParams.update({
    'font.size': 20,
    'font.family': 'Arial',
    'xtick.labelsize': 20,
    'ytick.labelsize': 20
})

In [2]:
# Set random seeds for reproducibility 
import glob
import numpy as np
import random
import torch
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

#### Data Loading

In [7]:
# Load CuP traces from .npy files and split them into train/val/test subsets with a fixed random seed
def load_and_split_traces(npy_path, label, seed=42):
    data = np.load(npy_path, allow_pickle=True)  
    traces = list(data)
    
    # Reproducible shuffling
    random.seed(seed)
    random.shuffle(traces)
    n_total = len(traces)
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    n_test = n_total - n_train - n_val
    train_samples = [(t, label) for t in traces[:n_train]]
    val_samples   = [(t, label) for t in traces[n_train:n_train + n_val]]
    test_samples  = [(t, label) for t in traces[n_train + n_val:]]
    return train_samples, val_samples, test_samples

cup_paths = glob.glob("../data/CuP-CAF/CuP-0.1mM_train_val_test.npy")
cup_train_samples = []
cup_val_samples = []
cup_test_samples = []

for path in cup_paths:
    # Label=0 for CuP class; use the same seed to keep splits consistent across runs
    train_part, val_part, test_part = load_and_split_traces(path, label=0, seed=42)
    cup_train_samples.extend(train_part)
    cup_val_samples.extend(val_part)
    cup_test_samples.extend(test_part)
    
# Quick sanity check of split sizes
print(f"CuP training trace count: {len(cup_train_samples)}")
print(f"CuP validation trace count: {len(cup_val_samples)}")
print(f"CuP test trace count: {len(cup_test_samples)}")

CuP training trace count: 2325
CuP validation trace count: 498
CuP test trace count: 499


#####  Truncation experiment on the test set.

In [25]:
# Optionally apply truncation during robustness testing, 
# then return time- and frequency-domain inputs.
from torch.utils.data import Dataset, DataLoader
class TraceDataset(Dataset):
    def __init__(self, samples, mode_select='train', seed=1024, robustness_type='none'):
        self.samples = samples
        self.mode_select = mode_select
        self.seed = seed
        self.robustness_type = robustness_type

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        trace, label = self.samples[idx]
        trace = np.array(trace, dtype=np.float32)

        if self.mode_select == 'robustness_test':
            if self.robustness_type == 'truncate_Au-Au_contact':
                # Truncating the Au-Au contact region
                trace = trace[250:]

            elif self.robustness_type == 'truncate_tunneling':
                # Truncating the tunneling background
                trace = trace[:-250]

            elif self.robustness_type == 'truncate_plateau':
                # Truncating the molecular plateau
                trace = np.concatenate([trace[:250], trace[750:]], axis=0)

            elif self.robustness_type == 'none':
                pass

            else:
                raise ValueError(
                    f"Unsupported robustness_type: {self.robustness_type}. "
                    f"Choose from ['none', 'truncate_contact', 'truncate_tunneling', 'truncate_plateau']."
                )

        # Time-domain input
        x_time = torch.tensor(trace, dtype=torch.float32).unsqueeze(0)

        # Frequency-domain input
        freq = np.fft.rfft(trace)
        mag = np.abs(freq)
        x_freq = torch.tensor(mag[np.newaxis, :], dtype=torch.float32)

        return x_time, x_freq, label

#### Model

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class TFC(nn.Module):
    def __init__(self, configs):
        super(TFC, self).__init__()
        
        # Time-domain encoder
        self.encoder_time = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=9, padding=4),
            nn.InstanceNorm1d(64, affine=True),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            
            # original code
            # Disable this block when running the random truncation robustness test.
#             nn.Flatten(),
#             nn.Linear(128 * 750, 128),
            
            # Updated for the random truncation robustness test: use global pooling to make the encoder length-invariant.
            # Disable this block when not running the random truncation robustness test.
            nn.AdaptiveAvgPool1d(1),  
            nn.Flatten(),       
            nn.Linear(128, 128),      
            
            nn.LayerNorm(128), 
            nn.GELU(),
            nn.Dropout(0.25)
        )
        
        # Frequency-domain encoder
        self.encoder_freq = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.InstanceNorm1d(64, affine=True),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            
            # original code
            # Disable this block when running the random truncation robustness test.
#             nn.Flatten(),
#             nn.Linear(128 * 375, 128),
            
            # Updated for the random truncation robustness test: use global pooling to make the encoder length-invariant.
            # Disable this block when not running the random truncation robustness test.
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, 128),
            
            nn.LayerNorm(128), 
            nn.GELU(),
            nn.Dropout(0.25)
        )

        # Projection heads
        self.projector_t = nn.Sequential(
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU()
        )
        
        self.projector_f = nn.Sequential(
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU()
        )

    def forward(self, x_in_t, x_in_f, mode="both"):
        # Return encoder features (h_*) and projected features (z_*)
        h_t, h_f = None, None
        z_t, z_f = None, None

        # Compute time branch if requested
        if mode in ["both", "time"]:
            h_t = self.encoder_time(x_in_t)  
            z_t = self.projector_t(h_t)       

        # Compute frequency branch if requested    
        if mode in ["both", "freq"]:
            h_f = self.encoder_freq(x_in_f)   
            z_f = self.projector_f(h_f)        

        return h_t, h_f, z_t, z_f

# classifier
class target_classifier(nn.Module):
    def __init__(self, configs):
        super(target_classifier, self).__init__()
        # classifier on concatenated (time, freq) features
        self.logits = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),       
            nn.Linear(128 * 2, 64), 
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, h_t, h_f):
         # In time-domain mode, use time-domain features
         # In frequency-domain mode, use frequency-domain features
         # In time-frequency fusion mode, use both time-domain and frequency-domain features
        if h_t is None: h_t = torch.zeros_like(h_f)
        if h_f is None: h_f = torch.zeros_like(h_t)
        
        h = torch.cat([h_t, h_f], dim=1) 
        out = self.logits(h)
        return out

##### Model Training (load model)

In [18]:
# Basic configs and device setup
class Config:
    TSlength_aligned_t = 1500
    TSlength_aligned_f = 751
    num_classes_target = 2

configs = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build model + classifier and load checkpoint weights
tfc_model = TFC(configs).to(device)
classifier = target_classifier(configs).to(device)

checkpoint = torch.load("checkpoint_both_truncation-experiment(CuP+CAF).pth")
tfc_model.load_state_dict(checkpoint['tfc_model_state_dict'])
classifier.load_state_dict(checkpoint['classifier_state_dict'])

# Switch to evaluation mode
tfc_model.eval()
classifier.eval()

target_classifier(
  (logits): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.25, inplace=False)
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [19]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay,classification_report,accuracy_score
from mpl_toolkits.axes_grid1 import make_axes_locatable
from collections import Counter
import matplotlib.colors as mcolors

# Evaluate the model
def evaluate(tfc_model, classifier, dataloader, mode="both", return_acc=False, save_path=None):
    tfc_model.eval()
    classifier.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x_t, x_f, y in dataloader:
            x_t, x_f = x_t.to(device), x_f.to(device)
            h_t, h_f, z_t, z_f = tfc_model(x_t, x_f, mode=mode)
            logits = classifier(h_t, h_f)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
            
    # Metrics summary
    print(f"\n[Mode={mode}] Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))
    acc = accuracy_score(all_labels, all_preds)
    print(f"[Mode={mode}] Accuracy: {acc:.4f}")

    # Plot confusion matrix only when meaningful (>=2 classes and not perfect accuracy)
    unique_labels = np.unique(all_labels)
    if (acc < 1.0) and (len(unique_labels) >= 2):
        cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

        with np.errstate(divide='ignore', invalid='ignore'):
            cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm_normalized = np.nan_to_num(cm_normalized)

        display_labels = ["CuP", "CuP+CAF"]
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Custom colormap for better contrast
        colors = ["#FFFFFF", "#F3A697", "#F17D65"]
        cmap_custom = mcolors.LinearSegmentedColormap.from_list("white_to_red", colors)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized,
                                      display_labels=display_labels)
        disp.plot(cmap=cmap_custom, values_format='.3f', ax=ax, colorbar=False)
        
        # Colorbar and axis styling
        im = disp.im_
        im.set_clim(0, 1)

        cbar = fig.colorbar(im, ax=ax)
        cbar.ax.tick_params(labelsize=20)
        cbar.ax.yaxis.label.set_size(22)

        plt.setp(ax.get_yticklabels(), ha="right", multialignment="center", fontsize=20)
        ax.set_xlabel("Predicted label", fontsize=22, fontweight='normal')
        ax.set_ylabel("True label", fontsize=22, fontweight='normal')
        plt.setp(ax.get_xticklabels(), ha='center', fontsize=20)
        plt.setp(ax.get_yticklabels(), rotation=90, va='center', fontsize=20)

        for text in disp.ax_.texts:
            text.set_fontsize(22)
            val = disp.confusion_matrix[int(text.get_position()[1])][int(text.get_position()[0])]
            text.set_color('white' if val > 0.5 else 'black')

        plt.tight_layout()

        # Save figure if requested
        if save_path:
            dir_name = os.path.dirname(save_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            plt.savefig(save_path, dpi=600, bbox_inches='tight')

        plt.show()

    else:
        if len(unique_labels) < 2:
            print("Only one true class is present; skipping confusion matrix plotting.")
        else:
            print("Accuracy is 100%; skipping confusion matrix plotting.")

    if return_acc:
        return acc

    
# Report the predicted class distribution over a dataloader
def predict_class_distribution(tfc_model, classifier, dataloader, device):
    tfc_model.eval()
    classifier.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_t, x_f, y in dataloader:
            x_t, x_f = x_t.to(device), x_f.to(device)
            h_t, h_f, z_t, z_f = tfc_model(x_t, x_f, mode=mode)
            logits = classifier(h_t, h_f)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    counter = Counter(all_preds)
    total = sum(counter.values())
    for cls_idx in sorted(counter.keys()):
        count = counter[cls_idx]
        percent = 100 * count / total
    return counter

In [20]:
# Load traces from multiple .npy files and attach a class label to each trace.
# Return a list of (trace, label) pairs.
def load_trace_level_samples(file_label_pairs):
    samples = []
    for path, label in file_label_pairs:
        data = np.load(path, allow_pickle=True) 
        for trace in data:
            trace = np.array(trace, dtype=np.float32)
            samples.append((trace, label))
    return samples

#   "both" = time-frequency fusion 
#   "time" = only-time-domain 
#   "freq" = only-frequency-domain
# Choose mode: "both"
mode = "both"

##### CuP-test（Full trace）

In [2]:
# Build CuP test set (force label=0)
# Run evaluation + prediction distribution check
final_test_samples = [(trace, 0) for trace, _ in cup_test_samples]
final_test_loader = DataLoader(TraceDataset(final_test_samples), batch_size=32, shuffle=False)

evaluate(tfc_model, classifier, final_test_loader, mode=mode)
predict_class_distribution(tfc_model, classifier, final_test_loader, device)

##### CuP-test (Truncation experiment )

In [1]:
# Monitor and report the average truncation length applied during truncation experiment
class StatsWrapper:
    def __init__(self, dataloader, original_len=1500):
        self.dataloader = dataloader
        self.original_len = original_len
        self.total_cut_len = 0
        self.total_samples = 0

    def __iter__(self):
        self.total_cut_len = 0
        self.total_samples = 0

        for batch in self.dataloader:
            x_t = batch[0]
            current_len = x_t.shape[-1]
            
            # Accumulate how much length was cut compared to the original length
            self.total_cut_len += (self.original_len - current_len)
            self.total_samples += 1

            yield batch

        if self.total_samples > 0:
            avg_cut = self.total_cut_len / self.total_samples
            print(f"\n>>> [Stats] Number of test samples: {self.total_samples}")
            print(f">>> [Stats] Average truncation length: {avg_cut:.2f} (~ {avg_cut/self.original_len:.1%})")

    def __len__(self):
        return len(self.dataloader)

# Build CuP test set (label=0) for robustness evaluation    
cup_final_samples = [(trace, 0) for trace, _ in cup_test_samples]

#### Truncating the Au-Au contact region

In [ ]:
# Dataset in robustness mode
cup_dataset = TraceDataset(
    cup_final_samples,
    mode_select='robustness_test',
    robustness_type='truncate_Au-Au_contact',
)

# Use batch_size=1 to measure per-trace truncation length accurately
cup_loader = DataLoader(cup_dataset, batch_size=1, shuffle=False)
cup_monitored_loader = StatsWrapper(cup_loader, original_len=1500)

evaluate(tfc_model, classifier, cup_monitored_loader, mode=mode)

#### Truncating the tunneling background

In [ ]:
# Dataset in robustness mode
cup_dataset = TraceDataset(
    cup_final_samples,
    mode_select='robustness_test',
    robustness_type='truncate_tunneling',
)

# Use batch_size=1 to measure per-trace truncation length accurately
cup_loader = DataLoader(cup_dataset, batch_size=1, shuffle=False)
cup_monitored_loader = StatsWrapper(cup_loader, original_len=1500)

evaluate(tfc_model, classifier, cup_monitored_loader, mode=mode)

#### Truncating the molecular plateau

In [ ]:
# Dataset in robustness mode
cup_dataset = TraceDataset(
    cup_final_samples,
    mode_select='robustness_test',
    robustness_type='truncate_plateau',
)

# Use batch_size=1 to measure per-trace truncation length accurately
cup_loader = DataLoader(cup_dataset, batch_size=1, shuffle=False)
cup_monitored_loader = StatsWrapper(cup_loader, original_len=1500)

evaluate(tfc_model, classifier, cup_monitored_loader, mode=mode)